# QTrack 2026 — Level 2: Grover search

**Tri-Co Hackathon · Bi-Co Quantum Computing Club**

**Mission:** Use a quantum phase oracle and diffusion to amplify one marked item among four candidates.

**Prerequisites:** Levels 0–1 recommended; 2-qubit gates and basis-state conventions.  
**Deliverable:** A completed `.ipynb` notebook with your code, results, and explanations.

Follow the numbered tasks, implement the `TODO` functions, and use the self-checks to test your work.


## Getting started

Run the next cell. In Google Colab, upload this notebook and choose **Runtime → Run all** after completing your tasks. The first run may install Python packages using the internet. After installation, all circuit simulations run locally on the CPU.

**Bit order convention:** Qiskit writes the most significant bit on the **left** of a bitstring. For example `"10"` means qubit 1 is `1` and qubit 0 is `0`. Our checks follow this convention.

**If you get `ModuleNotFoundError`:** rerun the installation cell or restart the notebook kernel and rerun.

In [90]:
# First run: installs dependencies only if missing (requires internet).
import sys, subprocess, importlib.util
if importlib.util.find_spec("qiskit") is None or importlib.util.find_spec("qiskit_aer") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit>=2.5,<3", "qiskit-aer>=0.17,<0.19", "numpy>=1.26,<3", "matplotlib>=3.8"])
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService
print("Qiskit ready. Free local simulation only; no IBM account or paid hardware required.")

Qiskit ready. Free local simulation only; no IBM account or paid hardware required.


### How to work
1. Read the task and its expected result.
2. Replace `return None` with your implementation.
3. Run the **self-check** underneath. A passing check verifies an observable property but is not a complete grading decision.
4. Answer the **Explain** questions in Markdown, using your own words.
5. Run the notebook top to bottom before exporting. Do not delete your code, plots or outputs.

## Problem

A tiny database has four labeled items: `00`, `01`, `10`, `11`. Exactly one is marked. A classical random guess succeeds with probability 1/4. In this two-qubit example, one Grover iteration can amplify the marked state to (ideally) probability 1.

Use **Qiskit bit ordering**, so `"01"` means qubit 0 is 1. You must support **all four** possible marked bitstrings.

**Worked example — uniform starting state:**

In [91]:
uniform = QuantumCircuit(2)
uniform.h([0,1])
print(Statevector.from_instruction(uniform).probabilities_dict())

{np.str_('00'): np.float64(0.2499999999999999), np.str_('01'): np.float64(0.2499999999999999), np.str_('10'): np.float64(0.2499999999999999), np.str_('11'): np.float64(0.2499999999999999)}


### Task 2A — Oracle

Flip the **phase**, not probability, of the marked state. A CZ gate with appropriate X-conjugation is enough.

In [92]:
def phase_oracle(marked: str) -> QuantumCircuit:
    """Create a 2-qubit circuit flipping ONLY the phase of the marked basis state."""
    qc = QuantumCircuit(2) # create 2 qubits that both start at 0
    # Flip any 0 bits to 1 so the marked state becomes 11
    if marked[0] == "0": 
        qc.x(1) # apply X gate to qubit 1
    if marked[1] == "0":
        qc.x(0) # apply X gate to qubit 0
    # Flip the phase of 11
    qc.cz(0, 1) # apply CZ gate with qubit 0 as control and qubit 1 as target
    # Undo the X gates to restore the original bits
    if marked[0] == "0":
        qc.x(1) # apply X gate to qubit 1
    if marked[1] == "0":
        qc.x(0) # apply X gate to qubit 0
    return qc

In [93]:
# Visible self-check (not a hidden grader)
if phase_oracle("01") is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    from qiskit.quantum_info import Operator
    diag = np.diag(Operator(phase_oracle('01')).data)
    assert np.allclose(diag, [1,-1,1,1])
    print("Self-check passed.")

Self-check passed.


### Task 2B — Diffusion

Construct inversion about the mean using H and X gates around a CZ.

In [94]:
def diffuser() -> QuantumCircuit:
    """Return the standard 2-qubit Grover diffusion operator (global phase irrelevant)."""
    circuit = QuantumCircuit(2)
    circuit.h([0, 1])
    circuit.x([0, 1])
    circuit.cz(0, 1)
    circuit.x([0, 1])
    circuit.h([0, 1])
    return circuit

In [95]:
# Visible self-check (not a hidden grader)
if diffuser() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    from qiskit.quantum_info import Operator
    circuit = diffuser()
    assert circuit.num_qubits == 2 and circuit.num_clbits == 0
    operator = Operator(circuit)
    assert np.allclose(operator.data.conj().T @ operator.data, np.eye(4))
    expected = Operator(0.5 * np.ones((4, 4)) - np.eye(4))
    assert operator.equiv(expected)
    print("Self-check passed.")

Self-check passed.


### Task 2C — One Grover iteration

Combine the preparation, oracle and diffuser. **Expected:** the target has probability almost 1 for every marked item.

In [96]:
def grover_search(marked: str) -> QuantumCircuit:
    """Prepare a uniform superposition, apply one oracle and one diffuser; no measurements."""
    uniform = QuantumCircuit(2)
    uniform.h([0,1])

    return uniform.compose(phase_oracle(marked).compose(diffuser()))

In [97]:
# Visible self-check (not a hidden grader)
if grover_search("10") is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(grover_search('10')).probabilities_dict()
    assert p.get('10',0)>.999999
    print("Self-check passed.")

Self-check passed.


### Task 2D — Check all targets

Test all four possible marked states; return a dictionary mapping each one to its success probability.

In [98]:
def success_table() -> dict:
    """Return a mapping from each of four marked states to its ideal success probability."""
    dict = {}

    for marked in ['00','01','10','11']:
        circuit = grover_search(marked)
        circuit.measure_all()

        backend = AerSimulator()

        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
        isa_circuit = pm.run(circuit)

        sampler = Sampler(mode=backend)

        job = sampler.run([isa_circuit], shots=1024)
        result = job.result()

        counts = result[0].data.meas.get_counts()

        probability = counts.get(marked, 0) / 1024

        dict[marked] = probability

    return dict

In [99]:
# Visible self-check (not a hidden grader)
if success_table() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    t = success_table()
    assert set(t)=={'00','01','10','11'} and min(t.values())>.999999
    print("Self-check passed.")

Self-check passed.


## Task 2E — Explain and validate (required, 120–220 words)

1. Why must the oracle flip **phase** rather than just permute basis states?
2. Why does one iteration suffice for N=4 with one marked item? (A conceptual answer is enough.)
3. Compare the classical random-guess baseline of 25% with your simulated success rates for all four targets.
4. Would this two-qubit demonstration alone establish a practical quantum speedup? Why or why not?

**Write your response below.**

In [100]:
if success_table() is not None:
    print('Classical one-guess baseline:',0.25)
    for state, p in success_table().items():
        print('Marked:',state,'Grover success:',round(p,6))
else:
    print('Finish tasks 2A–2D first.')

Classical one-guess baseline: 0.25
Marked: 00 Grover success: 1.0
Marked: 01 Grover success: 1.0
Marked: 10 Grover success: 1.0
Marked: 11 Grover success: 1.0


## Completion checklist
- [ ] A general phase oracle supports every 2-bit target
- [ ] Diffusion and full Grover circuit work
- [ ] All four target cases verified
- [ ] Required explanation is written
- [ ] Self-checks pass and notebook runs top to bottom

**Optional stretch:** Implement three-qubit Grover search with 1 and 2 iterations. Report how the target probability changes; document how your oracle marks an arbitrary 3-bit state.

## AI Use Disclosure
Task 2B:
- AI assistance implemented the H–X–CZ–X–H diffuser, strengthened its matrix self-check, and verified amplification for all four marked states.

## AI Use Disclosure
Task 2C:
- syntax for combining circuits
- troubleshooting

Task 2D:
- clarifying what the task is asking for
- learning how to extract from the dict
- learning how to create a dict